In [3]:
import cv2
import numpy as np
from jaad_data import JAAD
from random import randint
import os 
import pandas as pd
from time import sleep
from tqdm import tqdm
import json
import matplotlib.pyplot as plt

## calculate optimal lowerbound height and lowerbound width
1. pedestrian bbox height and width are expanded 1.5 times to capture surrounding objects
2. pedestrian bbox with new_height and new_width can still be too small for the masking model to mask humans
3. we stack a group of new_height to a list for all pedestrian bboxes (so does new_width)
4. we put two lists into separate boxplots to show size statistics 
5. the optimal lowerbound height/width is therefore the first quantile of the new_height boxplot or the new_width boxplot.


In [4]:
def show_boxplot_25_75(li, return_firstquantile=False): #shows the first and third quantile
    l = np.array(li)
    B = plt.boxplot(l)
    min_max = [item.get_ydata()[1] for item in B['whiskers']]
    medians = [item.get_ydata()[0] for item in B['medians']]
    first_third_quantile = [item.get_ydata()[0] for item in B['whiskers']]
    print('unit: a pixel of bbox width or height')
    print('min: {0}'.format(int(min_max[0])))
    print('max: {0}'.format(int(min_max[1])))
    print('first quantile (25%): {0}'.format(int(first_third_quantile[0])))
    print('second quantile (50%): {0}'.format(int(medians[0])))
    print('fourth quantile (75%): {0}'.format(int(first_third_quantile[1])))
    if return_firstquantile:
        return int(first_third_quantile[0])
    

In [5]:
cwd = os.getcwd()
jaad_path = cwd
pickle_path = jaad_path+"/data_cache/jaad_database.pkl"
dic = pd.read_pickle(pickle_path)
annotation = 'ped_annotations'
vids = list(dic.keys())

ratio = 1.5
old_widths = []
old_heights = []
new_widths = []
new_heights = []

### generate a boxplot for both unadjusted new_width and new_height 

In [6]:
height_lowerbound,width_lowerbound = 168, 67

## Output image data and a bbox size dictionary 
1. adjust new_height and new_width to lowerbounds
2. output surrounding bbox with .png format
3. output a dictionary called "bbox_beh.json/bbox_no_beh.json" to save info about the final height/width of a given surrounding bbox and pedestrian bbox. 

In [7]:
def adjust_bbox_size(new_height, new_width, height_lowerbound, width_lowerbound): #adjust new_height and new_width to at least have height_lowerbound, width_lowerbound size
    if new_height<height_lowerbound:
        adjust_ratio_1 = height_lowerbound/new_height
        adjust_height = height_lowerbound
        adjust_width = new_width*adjust_ratio_1
    elif new_width<width_lowerbound:
        adjust_ratio_2 = width_lowerbound/new_width
        adjust_width = width_lowerbound
        adjust_height = new_height*adjust_ratio_2        
    else:#no need to adjust because new_width and new_height are bigger than lowerbounds
        adjust_width, adjust_height = new_width, new_height
    return adjust_width, adjust_height


In [8]:
def find_last_occurance_index(li, element):
    li.reverse()
    index = li.index(element)
    return len(li) - index - 1

In [9]:
li = [0,0,1,1,1]
find_last_occurance_index(li,1)

4

### Extract surrounding boxes with behaviour labels

In [35]:
def saveimg_updatebbox(bbox_dict, ped, current_frame_index, c, x1, y1, x2, y2, new_x1, new_y1, new_x2, new_y2, cropped_bbox, save_bboxes_folder):
    f =  '{0}_{1}_c{2}.png'.format(str(ped), str(current_frame_index), str(c))
    if f not in bbox_dict[case].keys():
        bbox_dict[case][f] = {}
        bbox_dict[case][f]['ped_bbox'] = []
        bbox_dict[case][f]['surounding_bbox'] = []
    bbox_dict[case][f]['ped_bbox'] = [int(x1),int(y1),int(x2),int(y2)]
    bbox_dict[case][f]['surounding_bbox'] = [int(new_x1),int(new_y1),int(new_x2),int(new_y2)]
    cv2.imwrite(save_bboxes_folder + f, cropped_bbox)
    return bbox_dict

In [41]:
cwd = os.getcwd()
jaad_path = cwd
pickle_path = jaad_path+"/data_cache/jaad_database.pkl"
dic = pd.read_pickle(pickle_path)
annotation = 'ped_annotations'
vids = list(dic.keys())
bbox_dict = {'pos':{}, 'neg':{}}

adjust = True #adjust heights/widths to fit certain lowerbound
ratio = 1.5
new_widths = []
new_heights = []


save_data_1 = 'vid_images_surroundings_beh/'
save_data_2 = 'vid_images_surroundings_nobeh/'
save_both_data = [save_data_1, save_data_2]

for save_data_folder in save_both_data:
    if not os.path.exists(cwd+"/"+ save_data_folder):
        os.mkdir(cwd+"/"+save_data_folder)

        
for i in range(len(vids)): 
    vid = vids[i]
    vid_no = int(vid[-4:])
    save_vid_folder = str(vid)+'/'
    
    for save_data_folder in save_both_data:
        save_path = cwd+"/"+save_data_folder+save_vid_folder
        if not os.path.exists(save_path):
            os.mkdir(save_path)
            if not os.path.exists(save_path+'pos/'): #create pos/ and neg/
                os.mkdir(save_path+'pos/')
            if not os.path.exists(save_path+'neg/'):
                os.mkdir(save_path+'neg/')          


for i in tqdm(range(len(vids))): #tqdm show progress bar
    vid = vids[i]
    vid_no = int(vid[-4:])             
    save_vid = str(vid)+'/'    

    actual_height = dic[vid]['height']
    actual_width = dic[vid]['width']
    vid_path = jaad_path+"/JAAD_clips/"+vid+".mp4"
    peds = dic[vid][annotation].keys()
    cap = cv2.VideoCapture(vid_path)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, actual_width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, actual_height)
    trans_index = {}


    #identify pos/neg cases
    poss =[] #positive cases
    negs =[] # negative cases


    for ped in peds:
        trans_index[ped] = 0 #give every ped a trans index
        if ped[-1] == 'b':
            css = dic[vid][annotation][ped]['behavior']['cross']
            if 1 in css: #positive case: at least a frame has crossing stage, cs==1
                poss.append(ped)
            else: #negative case
                negs.append(ped)
        else:
            negs.append(ped)# samples in JAADbobeh are all with negative labels 



    while(True):
        ret, frame = cap.read()

        if ret == True:
            current_frame_index = int(cap.get(cv2.CAP_PROP_POS_FRAMES)-1)
            for i, ped in enumerate(peds):
                if ped[-1] == 'b':
                    save_vid_folder = save_data_1+save_vid
                else:
                    # save_vid_folder = None
                    save_vid_folder = save_data_2+save_vid
                    
                if ped in poss: #positive case
                    save_bboxes_folder = save_vid_folder + 'pos/'
                    case = 'pos'                                       

                elif ped in negs: #negative case
                    save_bboxes_folder = save_vid_folder + 'neg/'
                    case = 'neg'
                else:
                    save_bboxes_folder = None
                
                if current_frame_index in dic[vid][annotation][ped]['frames']:
                    if ped in poss or ped in negs:
                        bbox = dic[vid][annotation][ped]['bbox'][trans_index[ped]]
                        x1, y1, x2, y2 = bbox[0], bbox[1], bbox[2], bbox[3]
                        center_x,center_y = (x1+x2)/2, (y1+y2)/2
                        new_width,new_height = (x2-x1)*ratio,(y2-y1)*ratio
                        if adjust:#rescale to H/W_LB: if 1.5*old_H/W does not reach H/W_LB
                            new_width,new_height = adjust_bbox_size(new_height, new_width, height_lowerbound, width_lowerbound)
                        new_widths.append(int(new_width))
                        new_heights.append(int(new_height))
                        new_y1, new_y2, new_x1, new_x2 = max(int(center_y-(new_height/2)),0), min(int(center_y+(new_height/2)),1023), max(0,int(center_x-(new_width/2))), min(int(center_x+(new_width/2)),1919)
                        cropped_bbox = frame[new_y1:new_y2, new_x1:new_x2]
                        if ped[-1] == 'b':
                            crossing_li = list(dic[vid][annotation][ped]['behavior']['cross'])
                            c = crossing_li[trans_index[ped]] 
                            if 1 in crossing_li:
                                last_crossing_index = find_last_occurance_index(crossing_li, element=1)
                            else:
                                last_crossing_index = None
                        else:
                            last_crossing_index = None
                            c = 0 #for samples in JAADnobeh, every frame is no crossing
#                         o = dic[vid][annotation][ped]['occlusion'][trans_index[ped]]
    
                        if save_bboxes_folder:
                            if ped[-1] == 'b':
                                if last_crossing_index and last_crossing_index<=trans_index[ped]:
                                    bbox_dict = saveimg_updatebbox(bbox_dict, ped, current_frame_index, c, x1, y1, x2, y2, new_x1, new_y1, new_x2, new_y2, cropped_bbox, save_bboxes_folder)
                                else:
                                    bbox_dict = saveimg_updatebbox(bbox_dict, ped, current_frame_index, c, x1, y1, x2, y2, new_x1, new_y1, new_x2, new_y2, cropped_bbox, save_bboxes_folder)
                            else:
                                bbox_dict = saveimg_updatebbox(bbox_dict, ped, current_frame_index, c, x1, y1, x2, y2, new_x1, new_y1, new_x2, new_y2, cropped_bbox, save_bboxes_folder)
                        trans_index[ped] +=1

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        else:
            break           

    cap.release()
    cv2.destroyAllWindows()
save_json_name = 'bbox_beh_include_crossing.json'
a_file = open(save_json_name, "w")
json.dump(bbox_dict, a_file)
a_file.close()


100%|█████████████████████████████████████████| 346/346 [30:48<00:00,  5.34s/it]


In [28]:
df = load_dict('bbox_beh_include_crossing.json')
df2 = load_dict('bbox_beh_include_crossing_new.json')
print(len(list(df2.keys())), len(list(df.keys())))

391038 2


In [1]:
from utils.data_loader import load_dict
# df3 = load_dict("bbox_beh_include_crossing_withkeypoints_update.json")

In [3]:
df3 = load_dict("bbox_beh_include_crossing_withkeypoints_update_nobeh.json")

In [5]:
len(list(df3.keys()))

391038